# Análsis de la copa mundial femenina (Parte I)

## 0. Contexto y problema

En este cuaderno de `jupyter` se hace un análisis de los datos de los partidos de futbol de la copa mundial femenina.
El futbol femenino ha tenido un crecimiento en interés y audiencia en los últimos años, un análisis de estos datos puede ayudar a identificar tendencias de interés relacionados al desempeño de los equipos. 
Se hace un énfasis en aquellos equipos que han dominado históricamente y en las tendencias realacionadas al número de goles y puntuaciones a lo largo del torneo. 

La importancia de este análisis radica en responder a la pregunta de cómo está evolucionando el furbol femenino a lo largo de los años, a si existen equipos de futbol que se destacen por encima del resto o que siempre presentan un mal desempeño, y a visión general, si este deporte se está conviertiendo de un espacio cada vez más competitivo. 


## 1. Propuesta

Para poder identificar las tandencias de los datos obtenidos a lo largo de las últimas décadas, se ha optado por la construcción de un flujo de trabajo que inicie desde la captura de los datos de entrada hasta la visualización de los resultados. 
La gran ventaja de la construcción de este flujo es que permite una mejor comprensión de la solución puesto que ésta se encuentra separada en etapas.
Cada etapa es clave, cumple una función relevante y aporta valor al flujo de datos.
Adicionalmente, se ha construído una visualización interactiva en donde se presentan los resultados de análisis de los datos de entrada. 

![workflow](media/caso1/caso-01-mundial-femenino.drawio.png)

Adicionalmente, el flujo de trabajo integra herramientas de código abierto que hacen posible el funcionamiento del flujo de trabajo. Entre las herramientas seleccionadas destaca: 

- **python:** Python es el lenguaje de programación con el cual se implementan los paquetes para este flujo de trabajo se ejecute correctamente. Python se descata por su facilidad de uso, y la abundancia de paquetes relacionados con ciencia y análisis de datos. En este proyecto se implementarán los siguientes.
    - **jupyter**. Permite la creación de cuadernos interactivos (como éste) donde se combine código con mensajes. Esto facilita la comprensión de qué se desarrolló en el proyecto.
    - **pandas y numpy**. Paquetes que presenttan estructuras de datos y subturinas que facilitan el cálculo y extracción de patrones de los datos. 
    - **streamlit y metabase**. Permite visualizar los resultados del análsis de datos.
    - **matplotlib y seaborn**. Permite hacer visualizaciones durante cada etapa del procesamiento. 

- **Conda:** para administrar los entornos de trabajo y los paquetes de python necesarios del flujo de trabajo.

- **duckdb:** para el almacenamiento de datos procesados y sin procesar. 

- **Docker:** para desplegar la aplicación de visualización. 


![workflow](media/caso1/caso-01-mundial-femenino-architecture.drawio.png)

## 2. Implementación de la primera etapa

En esta primera etapa se descargan los datos a analizar, si se hace una inspección respecto a la existencia de valores nulos, duplicados. 

In [ ]:
import os
import sys


import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


from pynaowee.utils import download_file
from pynaowee.utils import display_categorical_values

In [ ]:
# algunas constantes importantes

DATA_URL_WCW = "https://raw.githubusercontent.com/daramireh/simonBolivarCienciaDatos/refs/heads/main/world_cup_women.csv"
DATA_URL_MATCHES = "https://raw.githubusercontent.com/daramireh/simonBolivarCienciaDatos/refs/heads/main/matches_1991_2023.csv"

DATASET_DIR = "./datasets/"
DATALAKEHOUSE_DIR = "./datalakehouse/"

WORLD_CUP_WOWEN_FILE = os.path.join(DATASET_DIR, "world-cup-women.csv")
WORLD_CUP_MATCHES_FILE = os.path.join(DATASET_DIR, "world-cup-matches.csv")

DUCKDB_FILE = os.path.join(DATALAKEHOUSE_DIR, "datalakehouse.duckdb")


### 2.1. Descarga del dataset

Para este proyecto se alamencenan los datos dentro de una instancia de duckdb.

![datawarehouse](media/caso1/caso-01-mundial-femenino-datawarehouse.drawio.png)

Se divide el entorno de duckdb en tres esquemas principales
- el esquema  **raw** almacena los datos tan como provienen de la fuente. 
- el equema **curate** hace modiciaciones de los datos del esquema anterior con el fin de hacer correcciones como: 
    - manejo de datos nulos
    - manejo de datos duplicados 
    - agregar indices u otras columnas "simples" 
- el esquema **insight** toma los datos del esquema anterior, hace las transformaciones pertinenetes y obtiene información de valor lista para ser visualizada. 

In [ ]:
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(DATALAKEHOUSE_DIR, exist_ok=True)

if not os.path.isfile(WORLD_CUP_WOWEN_FILE):
    download_file(DATA_URL_WCW, WORLD_CUP_WOWEN_FILE)

if not os.path.isfile(WORLD_CUP_MATCHES_FILE):
    download_file(DATA_URL_MATCHES, WORLD_CUP_MATCHES_FILE)

In [ ]:
### almacenar los datos originales en duckdb

with duckdb.connect(DUCKDB_FILE) as conn:
    
    conn.execute("CREATE SCHEMA IF NOT EXISTS s00_raw;")
    conn.execute("DROP TABLE IF EXISTS s00_raw.world_cup_women;")
    conn.execute("DROP TABLE IF EXISTS s00_raw.world_cup_matches;")
    
    # almacenar los datos originales
    conn.execute("""
        CREATE TABLE s00_raw.world_cup_women AS
        SELECT * FROM read_csv_auto(?);
    """, [WORLD_CUP_WOWEN_FILE])
    conn.execute("""
        CREATE TABLE s00_raw.world_cup_matches AS
        SELECT * FROM read_csv_auto(?);
    """, [WORLD_CUP_MATCHES_FILE])
    
    print(f"Datos originales almacenados en duckdb correctamente en {DUCKDB_FILE}.")

In [ ]:
### cargar los datos originales desde duckdb

df_wcw = None
df_wcm = None

with duckdb.connect(DUCKDB_FILE) as conn:

    df_wcw = conn.sql("SELECT * FROM s00_raw.world_cup_women;").df()
    df_wcm = conn.sql("SELECT * FROM s00_raw.world_cup_matches;").df()

    print("Datos cargados correctamente desde duckdb.")
    

### 2.2 Un vistazo a los datasets

El dataset se compone de dos archivos principalmente: 

- `world-cup-women.csv`. Este archivo contiene los resultados de los dos campeanatos mundiales de fúlbol femenino.
Debido a que los mundiales ocurren cada cuatro años, el archivo contien solamente 9 registros, que corresponden a los mundiales desde 1991 hasta 2023.
En otras palabras, cada fila es el resultado de un mundial. 
El dataset muestra la evolución del torneo a lo largo de las décadas, las selecciones más exitosas como Estados Unidos, Alemania y Japón, y las máximas goleadoras de cada edición, en formato nombre - número de goles.

- `world-cup-matches.csv`. Este archivo contiene los resultados de los partidos de los campeonatos mundiales de fútbol femenino. 
Este dataset contine muchas más columnas y registros que el anterior. 
Además de los datos básicos de un partido, como año, anfitrión, resultados y equipos enfrentados continee información de goles, tarjetas, susticiones, albitros, director ténico, entre otras. 


#### 2.2.1. `world-cup-women.csv` dataset

In [ ]:
df_wcw.info()

In [ ]:
# display_categorical_values(df_wcw_original)

In [ ]:
df_wcw.head()

#### Columnas del Dataset

| Columna | Tipo de Dato | Descripción |
|---------|--------------|-------------|
| **Year** | Numérico | Año en que se celebró el torneo |
| **Host** | Texto | País o países anfitriones del mundial |
| **Teams** | Numérico | Número de selecciones participantes |
| **Champion** | Texto | Selección campeona del torneo |
| **Runner-Up** | Texto | Selección subcampeona (segundo lugar) |
| **TopScorer** | Texto | Máxima goleadora del torneo con número de goles |
| **Attendance** | Numérico | Asistencia total de espectadores durante el torneo |
| **AttendanceAvg** | Numérico | Promedio de asistencia por partido |
| **Matches** | Numérico | Número total de partidos disputados |

#### 2.2.2. `world-cup-matches.csv` dataset

In [ ]:
df_wcm.info()

In [ ]:
# display_categorical_values(df_wcm_original)

In [ ]:
df_wcm.head()

| Columna | Tipo de Dato | Descripción |
|---------|--------------|-------------|
| home_team | String | Nombre del equipo local/casa |
| away_team | String | Nombre del equipo visitante |
| home_score | Integer | Goles anotados por el equipo local |
| home_xg | Float | Expected Goals (xG) del equipo local - métrica estadística de goles esperados |
| home_penalty | Integer | Número de penales cobrados por el equipo local (en definición por penales) |
| away_score | Integer | Goles anotados por el equipo visitante |
| away_xg | Float | Expected Goals (xG) del equipo visitante |
| away_penalty | Integer | Número de penales cobrados por el equipo visitante (en definición por penales) |
| home_manager | String | Nombre del entrenador/director técnico del equipo local |
| home_captain | String | Nombre de la capitana del equipo local |
| away_manager | String | Nombre del entrenador/director técnico del equipo visitante |
| away_captain | String | Nombre de la capitana del equipo visitante |
| Attendance | Integer | Número de asistentes al estadio |
| Venue | String | Nombre del estadio y ciudad donde se jugó el partido |
| Officials | String | Lista de árbitros y oficiales del partido (árbitro principal, asistentes, cuarto árbitro, VAR) |
| Round | String | Fase del torneo (Final, Semi-finals, Quarter-finals, Group stage, Round of 16, Third-place match) |
| Date | Date | Fecha del partido (formato: YYYY-MM-DD) |
| Score | String | Marcador final en formato texto (ej: "1–0", "2–1") |
| Referee | String | Nombre del árbitro principal |
| Notes | String | Notas especiales del partido (tiempo extra, penales, etc.) |
| Host | String | País(es) anfitrión(es) del mundial |
| Year | Integer | Año del mundial |
| home_goal | String | Lista de goleadoras del equipo local con minutos |
| away_goal | String | Lista de goleadoras del equipo visitante con minutos |
| home_goal_long | List[String] | Detalles completos de goles locales (minuto, marcador, jugadora, asistencia) |
| away_goal_long | List[String] | Detalles completos de goles visitantes (minuto, marcador, jugadora, asistencia) |
| home_own_goal | String | Autogoles a favor del equipo local |
| away_own_goal | String | Autogoles a favor del equipo visitante |
| home_penalty_goal | String | Goles de penal del equipo local (en tiempo regular/extra) |
| away_penalty_goal | String | Goles de penal del equipo visitante (en tiempo regular/extra) |
| home_penalty_miss_long | List[String] | Penales fallados por el equipo local con detalles |
| away_penalty_miss_long | List[String] | Penales fallados por el equipo visitante con detalles |
| home_penalty_shootout_goal_long | List[String] | Penales convertidos en tanda de penales por equipo local |
| away_penalty_shootout_goal_long | List[String] | Penales convertidos en tanda de penales por equipo visitante |
| home_penalty_shootout_miss_long | List[String] | Penales fallados en tanda de penales por equipo local |
| away_penalty_shootout_miss_long | List[String] | Penales fallados en tanda de penales por equipo visitante |
| home_red_card | String | Tarjetas rojas mostradas al equipo local |
| away_red_card | String | Tarjetas rojas mostradas al equipo visitante |
| home_yellow_red_card | String | Tarjetas amarillas-rojas (doble amarilla) al equipo local |
| away_yellow_red_card | String | Tarjetas amarillas-rojas al equipo visitante |
| home_yellow_card_long | List[String] | Detalles de tarjetas amarillas del equipo local (minuto, jugadora) |
| away_yellow_card_long | List[String] | Detalles de tarjetas amarillas del equipo visitante (minuto, jugadora) |
| home_substitute_in_long | List[String] | Sustituciones del equipo local (minuto, jugadora que entra, jugadora que sale) |
| away_substitute_in_long | List[String] | Sustituciones del equipo visitante (minuto, jugadora que entra, jugadora que sale) |




In [ ]:
# df_wcw = df_wcw_original.replace('None', np.nan)
# df_wcm = df_wcm_original.replace('None', np.nan)

In [ ]:
from ydata_profiling import ProfileReport

os.makedirs("eda-reports", exist_ok=True)
ProfileReport(df_wcw, title="World Cup Women Profiling Report").to_file("eda-reports/world-cup-women-report.html")
ProfileReport(df_wcm, title="World Cup Matches Profiling Report").to_file("eda-reports/world-cup-matches-report.html")

#### Concluisiones de los reportes generados

- `world-cup-women-report.html`
    - Faltan los datos de campeón y subcampeón del mundial de 2023. Estos datos pueden ser consulatos y posteriomente agregados.
    - Ha aumentado la cantidad de equipos y audiencia a lo largo de los años. 

- `world-cup-matches-report.html`
    - Las columnas `home_xg` y `away_xg` solo tienen datos para torneos recientes (2019-2023), ya que esta métrica es relativamente nueva
    - Muchas columnas contienen listas en formato string con delimitador `|` para múltiples eventos
    - Algunas columnas pueden estar vacías dependiendo del partido (ej: penales solo en partidos que llegaron a definición por penales)
    - No se presentan datos datos nulos en las columnas relevantes.
    - Los campos de `home_manager`, `away_manager`, `home_captain` y `away_captain` aparecen a partir de 2015.
    - Existen partidos con attendance 0



### Completando datos faltantes:

In [ ]:
# https://es.wikipedia.org/wiki/Copa_Mundial_Femenina_de_F%C3%BAtbol_de_2023

df_wcw.loc[df_wcw['Year'] == 2023, 'Winner'] = 'Spain'
df_wcw.loc[df_wcw['Year'] == 2023, 'Runners-Up'] = 'England'

In [ ]:
df_wcw.sort_values(by=['Year'], ascending=True, inplace=True)
df_wcw.reset_index(drop=True, inplace=True)

In [ ]:
# df_wcw["TopScorrer-Name"] = df_wcw["TopScorrer"].str.split(' - ').str[0]
# df_wcw["TopScorrer-Goals"] = df_wcw["TopScorrer"].str.split(' - ').str[1].astype(int)
# del df_wcw["TopScorrer"] 

In [ ]:
df_wcm.sort_values(by=['Date'], ascending=True, inplace=True)
df_wcm.reset_index(drop=True, inplace=True)
df_wcm['id'] = df_wcm.index + 1

In [ ]:
with duckdb.connect(DUCKDB_FILE) as conn:
    
    conn.execute("CREATE SCHEMA IF NOT EXISTS s01_curated;")
    conn.execute("DROP TABLE IF EXISTS s01_curated.world_cup_women;")
    conn.execute("DROP TABLE IF EXISTS s01_curated.world_cup_matches;")


    conn.execute("CREATE TABLE s01_curated.world_cup_women AS SELECT * FROM df_wcw;")
    conn.execute("CREATE TABLE s01_curated.world_cup_matches AS SELECT * FROM df_wcm;")

    print("datos almacenados correctamente en la zona curada.")

### Identificacion de la validacion curada. 

La tabla `world_cup_women` sintetiza la información de los mundiales femeninos, mientras que la tabla `world_cup_matches` muestra información más detallada.


#### Campos que relacionan las tablas
1. Year: que corresponde al año del mundial
2. Host: país o paises anfitriones del mundial. 
3. Champion y Runner-Up: corresponden a los paises campeon y subcampeón de cada torneo. En la tabla de matches, es posible identificar si un juego es una final con la columna `Round`, a partir de allí y de los resultados del partido se pueden generar estos datos. 
4. Matches: corresponde a hacer un conteo agrupado por años de los partidos juegados.
5. Attended: corresponde a hacer una suma agrupada de los participantes en cada encuentro.
6. Top-scorrer: corresponde a hacer una "desnormalización de los goles" para luego agrupar por año y juegador.  